# Feature Ablation & Final Forecasting Benchmark

This notebook evaluates whether the advanced feature engineering from Notebook 09 improves out-of-sample forecasting versus the baseline feature set used in Notebook 08.

The comparison uses the same rolling-origin validation logic and fixed model specifications. No hyperparameter tuning is performed on the evaluation folds.

## 1. Methodology

**Target:** next-year municipality price growth (`target_price_growth`).

**Validation:** for each forecast year, train on all earlier years and predict the forecast year. This preserves temporal ordering and avoids random train/test leakage.

**Models:**
- Dummy mean baseline;
- Ridge regression (`alpha=10`);
- HistGradientBoostingRegressor with the fixed specification used in Notebook 08.

**Comparison:** baseline features versus the advanced features generated by Notebook 09. The main metric is MAE; RMSE and R² are secondary diagnostics.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists(): PROJECT_ROOT = PROJECT_ROOT.parent
FEATURE_PATH = PROJECT_ROOT / 'data' / 'processed' / 'forecasting_features.csv'
if not FEATURE_PATH.exists():
    raise FileNotFoundError(
        f'{FEATURE_PATH} not found. Execute Notebook 09_advanced_feature_engineering.ipynb first to generate the processed forecasting dataset.'
    )

df = pd.read_csv(FEATURE_PATH)
df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
df['municipality_code'] = df['municipality_code'].astype('string')
print(f'Rows: {len(df):,} | Years: {df["year"].min()}–{df["year"].max()}')

## 2. Feature sets

The baseline reproduces the compact lag/level information used in the earlier forecasting benchmark. The advanced set is taken from Notebook 09 and adds temporal dynamics, rolling history, transaction intensity, logarithmic scale and regional context.

In [ ]:
baseline_features=[
    'price_growth_lag1','ntn_growth_lag1','population_growth_lag1',
    'price_level_lag1','ntn_level_lag1','population_level_lag1'
]

advanced_features=[
    'price_growth','ntn_growth','population_growth','ntn_per_1000',
    'price_growth_lag1','price_growth_lag2',
    'ntn_growth_lag1','ntn_growth_lag2',
    'population_growth_lag1','population_growth_lag2',
    'price_momentum_1y','ntn_momentum_1y','population_momentum_1y',
    'price_growth_roll2_mean','price_growth_roll3_mean','price_growth_roll3_std',
    'ntn_growth_roll2_mean','ntn_growth_roll3_mean','ntn_growth_roll3_std',
    'population_growth_roll2_mean','population_growth_roll3_mean','population_growth_roll3_std',
    'ntn_per_1000_lag1','ntn_per_1000_lag2',
    'log_price_lag1','log_ntn_lag1','log_population_lag1',
    'region_price_growth_lag1','region_ntn_growth_lag1','region_population_growth_lag1',
    'municipality_minus_region_price_growth_lag1','municipality_minus_region_ntn_growth_lag1'
]

required=['year','municipality_code','target_price_growth'] + baseline_features + advanced_features
missing=[c for c in required if c not in df.columns]
if missing:
    raise KeyError(f'Missing expected Notebook 09 columns: {missing}')

model_df=df.dropna(subset=['year','target_price_growth']).copy()
print(f'Model rows with target: {len(model_df):,}')
print(f'Baseline features: {len(baseline_features)} | Advanced features: {len(advanced_features)}')

## 3. Fixed model specifications

The models are deliberately kept identical across feature sets. This makes the comparison an ablation study rather than a hyperparameter optimization exercise.

In [ ]:
def make_model(model_name):
    if model_name == 'Ridge':
        return Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('model', Ridge(alpha=10.0)),
        ])
    if model_name == 'HistGradientBoosting':
        return Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('model', HistGradientBoostingRegressor(
                max_iter=150, learning_rate=0.05, max_leaf_nodes=15,
                l2_regularization=1.0, random_state=42
            )),
        ])
    if model_name == 'Dummy':
        return Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('model', DummyRegressor(strategy='mean')),
        ])
    raise ValueError(model_name)

models=['Ridge','HistGradientBoosting']
feature_sets={'Baseline':baseline_features,'Advanced':advanced_features}

## 4. Rolling-origin benchmark

For every forecast year with at least three distinct training years, the model is fitted only on observations from earlier years. The same forecast years are used for both feature sets.

In [ ]:
years=sorted(model_df['year'].dropna().astype(int).unique())
results=[]

for forecast_year in years:
    train=model_df[model_df['year'] < forecast_year]
    test=model_df[model_df['year'] == forecast_year]
    if train['year'].nunique() < 3 or test.empty:
        continue

    y_train=train['target_price_growth']
    y_test=test['target_price_growth']

    for feature_name, features in feature_sets.items():
        X_train=train[features]
        X_test=test[features]
        for model_name in models:
            model=make_model(model_name)
            model.fit(X_train,y_train)
            pred=model.predict(X_test)
            results.append({
                'forecast_year':forecast_year,
                'feature_set':feature_name,
                'model':model_name,
                'n_train':len(train),
                'n_test':len(test),
                'mae':mean_absolute_error(y_test,pred),
                'rmse':np.sqrt(mean_squared_error(y_test,pred)),
                'r2':r2_score(y_test,pred),
            })

# Dummy benchmark is feature-independent and is reported once per forecast year.
for forecast_year in years:
    train=model_df[model_df['year'] < forecast_year]
    test=model_df[model_df['year'] == forecast_year]
    if train['year'].nunique() < 3 or test.empty: continue
    pred=np.repeat(train['target_price_growth'].mean(),len(test))
    results.append({
        'forecast_year':forecast_year,'feature_set':'Reference','model':'Dummy',
        'n_train':len(train),'n_test':len(test),
        'mae':mean_absolute_error(test['target_price_growth'],pred),
        'rmse':np.sqrt(mean_squared_error(test['target_price_growth'],pred)),
        'r2':r2_score(test['target_price_growth'],pred),
    })

results_df=pd.DataFrame(results)
display(results_df.head(10))
print(f'Completed {results_df["forecast_year"].nunique()} forecast folds.')

## 5. Aggregate performance

A model should not be selected because of one unusually good forecast year. The table below summarizes mean and median performance across rolling folds.

In [ ]:
summary=(results_df[results_df['model']!='Dummy']
          .groupby(['model','feature_set'])
          .agg(folds=('forecast_year','nunique'),
               mean_mae=('mae','mean'),median_mae=('mae','median'),
               mean_rmse=('rmse','mean'),median_rmse=('rmse','median'),
               mean_r2=('r2','mean'),median_r2=('r2','median'))
          .reset_index()
          .sort_values(['model','mean_mae']))
display(summary)

## 6. Advanced-feature improvement

Positive MAE improvement means that the advanced feature set has lower error than the baseline. We also calculate the share of forecast years in which advanced features win, which is a simple stability diagnostic.

In [ ]:
paired=results_df[results_df['model']!='Dummy'].pivot_table(
    index=['forecast_year','model'],columns='feature_set',values=['mae','rmse','r2'])

paired['mae_improvement'] = 1 - paired['mae']['Advanced'] / paired['mae']['Baseline']
paired['rmse_improvement'] = 1 - paired['rmse']['Advanced'] / paired['rmse']['Baseline']
paired['r2_change'] = paired['r2']['Advanced'] - paired['r2']['Baseline']

improvement_summary=(paired.reset_index().groupby('model')
    .agg(mean_mae_improvement=('mae_improvement','mean'),
         median_mae_improvement=('mae_improvement','median'),
         share_years_advanced_better=('mae_improvement',lambda s:(s>0).mean()),
         mean_rmse_improvement=('rmse_improvement','mean'),
         mean_r2_change=('r2_change','mean'))
    .reset_index())
display(improvement_summary)

## 7. MAE by forecast year

The year-by-year view shows whether the advanced feature set improves forecasts consistently or only in selected periods.

In [ ]:
for model_name in models:
    plot_df=results_df[results_df['model']==model_name].pivot(index='forecast_year',columns='feature_set',values='mae')
    ax=plot_df[['Baseline','Advanced']].plot(marker='o',figsize=(9,4))
    ax.set_title(f'{model_name}: Rolling-origin MAE')
    ax.set_xlabel('Forecast year')
    ax.set_ylabel('MAE — next-year price growth')
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

## 8. Model-selection rule

For portfolio purposes, the preferred feature set is the one with lower mean MAE **and** evidence of stability across folds. A positive average improvement with a very low win rate should be treated cautiously.

This is a decision heuristic, not a formal statistical significance test. Hyperparameter tuning should be performed only with a time-aware inner validation scheme; that is a natural next step if additional forecasting work is needed.

In [ ]:
for model_name in models:
    row=improvement_summary.loc[improvement_summary['model'].eq(model_name)].iloc[0]
    decision=(row['mean_mae_improvement']>0 and row['share_years_advanced_better']>=0.5)
    verdict='ADVANCED' if decision else 'BASELINE'
    print(f'{model_name}: recommended feature set = {verdict} | mean MAE improvement={row["mean_mae_improvement"]:.2%} | win rate={row["share_years_advanced_better"]:.1%}')

## 9. Interpretation and limitations

- This is an **out-of-sample predictive benchmark**, not a causal analysis.
- OMI quotations are market reference values and are not equivalent to observed transaction prices.
- Municipality observations are spatially dependent, so standard fold metrics do not eliminate spatial correlation.
- The historical time span is short relative to the richness of the feature set; this increases the risk of overfitting.
- Regional aggregates are descriptive contextual predictors, not causal controls.
- The current benchmark deliberately avoids hyperparameter tuning on the evaluation folds.

The main portfolio conclusion should therefore focus on whether feature engineering produces a **repeatable out-of-sample gain**, rather than claiming that the advanced variables explain price dynamics causally.